In [1]:
import sys 
import xarray as xr
import numpy as np
import os
import pandas as pd
import yaml
import math
from datetime import datetime, timedelta
from parcels import FieldSet, Field, VectorField, ParticleSet, JITParticle, ParcelsRandom, Variable,StatusCode,GeographicPolar, Geographic
from glob import glob
import matplotlib.pyplot as plt

import warnings

warnings.filterwarnings(
    "ignore",
    message="The specified chunks separate the stored chunks",
    category=UserWarning,
)

warnings.filterwarnings(
    "ignore",
    message="File .* could not be decoded properly by xarray",
    category=UserWarning,
)

In [2]:
sys.path.append(os.getcwd()+'/Source')

from OP_Kernels import *
from OP_functions import *

In [3]:
def DebugJump(particle, fieldset, time):
    if particle.status == 1 and (particle_ddepth > 20 or particle_ddepth < -20):
        kz = fieldset.vert_eddy_diff[time, particle.depth, particle.lat, particle.lon, particle]
        td = fieldset.totaldepth[time, particle.depth, particle.lat, particle.lon, particle]
        (u, v, w) = fieldset.UVW[time, particle.depth, particle.lat, particle.lon, particle]
        print("JUMP ddepth=%g depth=%g kz=%g td=%g w=%g fact=%g" %
              (particle_ddepth, particle.depth, kz, td, w, particle.fact))

In [4]:
# run_debug.py
import sys, math
import numpy as np
import xarray as xr
from datetime import timedelta

sys.path.append('/home/jvalenti/MOAD/analysis-jose/notebooks/MP_model/Source')
from OP_functions import load_config
from run_Parcels import run_Parcels

cfg = 'yaml/config.yaml'
param = load_config(cfg)
P = param['particle']

# release points of the 3 bad trajectories, from the original run
ds = xr.open_zarr('/home/jvalenti/MOAD/results/test_function.zarr').load()
bad = np.where(ds.z.max('obs').values > 450)[0]
ncopy = 50

lon = np.repeat(ds.lon.isel(obs=0).values[bad], ncopy).astype(float)
lat = np.repeat(ds.lat.isel(obs=0).values[bad], ncopy).astype(float)
dep = np.repeat(ds.z.isel(obs=0).values[bad], ncopy).astype(float)
N = len(lon)

# a, b, c from the yaml means (sorted: a largest). The originals were random draws
# and were not written to output, so these are representative values.
c_, b_, a_ = np.sort([P['dim1'], P['dim2'], P['dim3']])
e = P['shape']
fe = e**2 * math.gamma(e/2)**3 / math.gamma(3*e/2)
a, b, c = np.full(N, a_), np.full(N, b_), np.full(N, c_)

particles = dict(lon=lon, lat=lat, depth=dep, a=a, b=b, c=c,
                 Vol=(2/3)*fe*a*b*c,
                 rho_p=np.full(N, float(P['density'])),
                 SI=np.ones(N))

run_Parcels(cfg,
            runtime=timedelta(hours=8),
            outputdt=timedelta(seconds=param['param']['dt']),
            outfile='/home/jvalenti/MOAD/results/debug_jump.zarr',
            particles=particles,
            extra_kernels=[DebugJump],
            convert=False,          # skip the netcdf conversion
            seed=1234)

/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/xarray/backends/plugins.py:80: RuntimeWarning: Engine 'erddapy' loading failed:
cannot import name 'T_PathFileOrDataStore' from 'xarray.backends.common' (/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/xarray/backends/common.py)
  warnings.warn(f"Engine {name!r} loading failed:\n{ex}", RuntimeWarning)


The model will run for 1 days, starting at 2023-01-02 00:00:00
INFO: Compiled ParcelsRandom ==> /tmp/parcels-2894/parcels_random_76ad3d0b-53f2-4897-b1c3-ff2be51342fa.c


/ocean/jvalenti/MOAD/analysis-jose/.pixi/envs/parcels/lib/python3.11/site-packages/parcels/kernel.py:254: KernelWarning: td declared in multiple Kernels
  kernel_ccode = kernelgen.generate(deepcopy(self.py_ast), self.funcvars)


INFO: Output files are stored in /home/jvalenti/MOAD/results/debug_jump.zarr.
 56%|█████▋    | 16200.0/28800.0 [00:38<00:28, 448.28it/s]JUMP ddepth=-20.3279 depth=32.539 kz=4.00887e-05 td=22.375 w=0 fact=0.956897
 62%|██████▎   | 18000.0/28800.0 [00:39<00:19, 563.96it/s]JUMP ddepth=-21.7376 depth=33.2439 kz=7.55652e-05 td=22.375 w=0 fact=0.948546
 69%|██████▉   | 19800.0/28800.0 [00:45<00:20, 439.49it/s]JUMP ddepth=-33.7205 depth=39.2352 kz=0 td=22.375 w=0 fact=0.971096
JUMP ddepth=-24.4746 depth=50.7402 kz=0.000282357 td=38.5 w=0 fact=0.973438
JUMP ddepth=-23.7751 depth=34.2626 kz=8.42648e-05 td=22.375 w=0 fact=0.953066
JUMP ddepth=-21.118 depth=32.934 kz=7.80078e-05 td=22.375 w=0 fact=0.947822
 75%|███████▌  | 21600.0/28800.0 [00:46<00:12, 572.12it/s]JUMP ddepth=-28.9185 depth=36.8343 kz=2.98071e-05 td=22.375 w=0 fact=0.964886
 81%|████████▏ | 23400.0/28800.0 [00:52<00:11, 462.29it/s]JUMP ddepth=-33.6974 depth=39.2238 kz=0 td=22.375 w=0 fact=0.980805
 88%|████████▊ | 25200.0/28800.0 

'/home/jvalenti/MOAD/results/debug_jump.zarr'

In [5]:
ds, name = zarr_tonet('/home/jvalenti/MOAD/results/debug_jump.zarr')
print(ds.status.isel(obs=-1).to_series().value_counts())   # fate of particles at the end
print(float(ds.ws.max()), float(ds.ws.min()))                # settling velocities
print(ds.N_hb.max().values, ds.N_ph.max().values)            # fouling counts
print(ds.z.min().values, ds.z.max().values)
print(int(ds.lon.isnull().sum()), int(ds.lat.isnull().sum()))  # NaNs = deleted/lost particles

status
1.0    100
2.0     35
3.0     15
Name: count, dtype: int64
0.00025894056307151914 0.0002240220201201737
3817.9834 0.98375106
0.0 78.98703341520233
0 0
